# Modeling

This notebook demonstrates the modeling process:
- Model training (Random Forest, XGBoost)
- Model evaluation
- Feature importance analysis
- Optimization recommendations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.utils import load_data
from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.train_model import ModelTrainer
from src.evaluate_model import ModelEvaluator
from src.optimization import ChargingOptimizer

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Load and preprocess data
df = load_data("ev_charging_data.csv")
preprocessor = DataPreprocessor()
df_processed = preprocessor.preprocess(df)

# Split data
train_df, test_df = preprocessor.time_based_split(df_processed, test_size=0.2)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

In [ ]:
# Feature engineering
feature_engineer = FeatureEngineer()

train_df_features = feature_engineer.engineer_features(train_df.copy(), encode_categorical=True)
test_df_features = feature_engineer.engineer_features(test_df.copy(), train_df=train_df, encode_categorical=True)

# Get features
feature_cols = feature_engineer.get_feature_columns(train_df_features, target_col='Charging Cost (USD)')

X_train = train_df_features[feature_cols].fillna(0)
y_train = train_df_features['Charging Cost (USD)']

X_test = test_df_features[feature_cols].fillna(0)
y_test = test_df_features['Charging Cost (USD)']

print(f"Features: {len(feature_cols)}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Train models
trainer = ModelTrainer()
models = trainer.train_all_models(X_train, y_train, tune_hyperparameters=False)

print(f"\nBest model: {trainer.best_model_name}")

In [ ]:
# Evaluate models
evaluator = ModelEvaluator()

# Evaluate best model
best_model = trainer.best_model
y_pred_test = best_model.predict(X_test)

test_metrics = evaluator.compute_metrics(y_test, y_pred_test, set_name="test")

# Visualizations
evaluator.plot_predictions_vs_actual(y_test, y_pred_test)
evaluator.plot_error_distribution(y_test, y_pred_test)

In [ ]:
# Feature importance
importance_df = trainer.get_feature_importance(top_n=20)
print("Top 20 Feature Importance:")
print(importance_df)

evaluator.plot_feature_importance(importance_df)

In [ ]:
# Optimization example
optimizer = ChargingOptimizer(best_model, feature_cols, feature_engineer)

# Get a sample session
sample_session = test_df_features.iloc[[0]].copy()

# Generate recommendations
recommendations = optimizer.generate_recommendations(sample_session)

print("Optimization Recommendations:")
print(f"Current Cost: ${recommendations['current_predicted_cost']:.2f}")
print(f"Optimal Cost: ${recommendations['optimal_predicted_cost']:.2f}")
print(f"Potential Savings: ${recommendations['potential_savings']:.2f}")
print(f"Savings Percentage: {recommendations['savings_percentage']:.1f}%")

In [ ]:
# Identify inefficient sessions
inefficient = optimizer.identify_inefficient_sessions(test_df_features)
print(f"\nFound {len(inefficient)} inefficient sessions")
print(f"Average potential savings: ${inefficient['potential_savings'].mean():.2f}")